In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
data = pd.read_csv("train.csv")
data['date'] = pd.to_datetime(data['date'])
data = data.sort_values(['store','item','date'])


In [3]:
data['sales_log'] = np.log1p(data['sales'])


In [4]:
# Lag features
data['lag_1'] = data.groupby(['store','item'])['sales_log'].shift(1)
data['lag_7'] = data.groupby(['store','item'])['sales_log'].shift(7)
data['lag_30'] = data.groupby(['store','item'])['sales_log'].shift(30)
data['lag_365'] = data.groupby(['store','item'])['sales_log'].shift(365)

# Rolling features
data['rolling_7'] = data.groupby(['store','item'])['sales_log']\
    .transform(lambda x: x.shift(1).rolling(7).mean())

data['rolling_30'] = data.groupby(['store','item'])['sales_log']\
    .transform(lambda x: x.shift(1).rolling(30).mean())

# Calendar
data['month'] = data['date'].dt.month
data['day_of_week'] = data['date'].dt.dayofweek

data = data.dropna()


In [5]:
data = pd.get_dummies(data, columns=['store','item'], drop_first=True)


In [6]:
split_date = data['date'].quantile(0.8)

train = data[data['date'] <= split_date]
valid = data[data['date'] > split_date]


In [7]:
drop_cols = ['date', 'sales', 'sales_log']
features = [col for col in data.columns if col not in drop_cols]

X_train = train[features]
y_train = train['sales_log']

X_valid = valid[features]
y_valid = valid['sales_log']


In [24]:
model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.01,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred_log = model.predict(X_valid)


In [25]:
predictions = np.expm1(pred_log)
actual = np.expm1(y_valid)

mae = mean_absolute_error(actual, predictions)
rmse = np.sqrt(mean_squared_error(actual, predictions))

print("Improved Panel MAE:", mae)
print("Improved Panel RMSE:", rmse)


Improved Panel MAE: 6.466695611973331
Improved Panel RMSE: 8.41085476352152
